In [3]:
  # Install the library to parse UD data
  !pip install conllu

In [4]:
# Download and Parse Data
import requests
from conllu import parse

def load_ud_data(url):
    print(f"Downloading from {url}...")
    response = requests.get(url)

    # Check if the download worked (Status 200 = OK)
    if response.status_code != 200:
        raise ValueError(f"Failed to download! Status code: {response.status_code}")

    return parse(response.text)

# We use 'train-a' because the full file was split due to size
train_url = "https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-train-a.conllu"
test_url = "https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-test.conllu"

raw_train_data = load_ud_data(train_url)
raw_test_data = load_ud_data(test_url)

print(f"Loaded {len(raw_train_data)} training sentences.")
print(f"Loaded {len(raw_test_data)} testing sentences.")

Loaded 24516 training sentences.
Loaded 8800 testing sentences.


In [15]:
import nltk
import requests
from nltk.tag import UnigramTagger
from conllu import parse

print("Downloading data...")
def load_ud_data(url):
    response = requests.get(url)
    return parse(response.text)

train_url = "https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-train-a.conllu"
test_url = "https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-test.conllu"

raw_train = load_ud_data(train_url)
raw_test = load_ud_data(test_url)

# Format Data for NLTK
print("Formatting data...")
def prepare_data(ud_data):
    # Convert to [[(Word, Tag), ...], ...] format
    return [[(t['form'], t['upos']) for t in sent] for sent in ud_data]

train_set_uni = prepare_data(raw_train)
test_set_uni = prepare_data(raw_test)

# Train the Model
print("Training Unigram Tagger...")
# This effectively creates a Python dictionary: {Word: Most_Frequent_Tag}
unigram_tagger = UnigramTagger(train_set_uni)

accuracy = unigram_tagger.accuracy(test_set_uni)
print(f"Unigram Accuracy: {accuracy:.2%}")

print("-" * 30)
# Test Sentence: "Мама мыла раму" (Mom washed the frame)
# Ambiguity Check: "мыла" is the past tense of "wash" (VERB),
# but "soap" (NOUN) is more common.
custom_sentence = "Мама мыла раму"
tokens = custom_sentence.split()
tags = unigram_tagger.tag(tokens)

print(f"Input: {custom_sentence}")
for word, tag in tags:
    # If the word isn't in the dictionary, Unigram gives 'None'
    status = tag if tag else "UNKNOWN (None)"
    print(f"{word:<10} -> {status}")

Formatting data...
Training Unigram Tagger...
Unigram Accuracy: 83.87%
------------------------------
Input: Мама мыла раму
Мама       -> NOUN
мыла       -> NOUN
раму       -> UNKNOWN (None)


In [6]:
# Train and Evaluate HMM Tagger
import nltk
from nltk.tag import HiddenMarkovModelTagger
import warnings

# 1. Function to convert data to NLTK format: [(Word, Tag), (Word, Tag)...]
def convert_to_nltk_format(ud_data):
    nltk_data = []
    for sentence in ud_data:
        sentence_tuples = []
        for token in sentence:
            # We use 'upos' (Universal POS) as the target label
            sentence_tuples.append((token['form'], token['upos']))
        nltk_data.append(sentence_tuples)
    return nltk_data

# 2. Convert the data
print("Formatting data...")
train_set = convert_to_nltk_format(raw_train_data)
test_set = convert_to_nltk_format(raw_test_data)

# 3. Train the HMM
print("Training HMM...")
warnings.filterwarnings("ignore")
hmm_tagger = HiddenMarkovModelTagger.train(train_set)
print("Training COMPLETE!")

# 4. Evaluation
print("-" * 30)
print("Calculating Accuracy...")
accuracy = hmm_tagger.accuracy(test_set)
print(f"HMM Model Accuracy: {accuracy:.2%}")

# 5. Inference (Test on Custom Sentence)
print("-" * 30)
# Example: "Mama myla ramu" (Mom washed the frame)
custom_sentence = "Мама мыла раму"
tokens = custom_sentence.split()

# The tagger expects a list of words
tags = hmm_tagger.tag(tokens)

print(f"Test Sentence: {custom_sentence}")
for word, tag in tags:
    print(f"{word:<10} -> {tag}")

Formatting data...
Training HMM...
Training COMPLETE!
------------------------------
Calculating Accuracy...
HMM Model Accuracy: 87.38%
------------------------------
Test Sentence: Мама мыла раму
Мама       -> NOUN
мыла       -> NOUN
раму       -> PUNCT


In [8]:
# CELL 6: Install CRF Library
!pip install sklearn-crfsuite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 13.1 MB/s eta 0:00:00


In [9]:
# CELL 7: Feature Extraction
import sklearn_crfsuite
from sklearn_crfsuite import metrics

def word2features(sentence, i):
    word = sentence[i][0]
    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],       # Last 3 letters (Crucial for Russian suffixes)
        'word[-2:]': word[-2:],       # Last 2 letters
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
    }
    # Context: Look at the PREVIOUS word
    if i > 0:
        word1 = sentence[i-1][0]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
        })
    else:
        features['BOS'] = True # Beginning of Sentence
    # Context: Look at the NEXT word
    if i < len(sentence)-1:
        word1 = sentence[i+1][0]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
        })
    else:
        features['EOS'] = True # End of Sentence

    return features

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [label for token, label in sent]

# Prepare the data (This converts your existing data into features)
print("Extracting features...")
X_train = [sent2features(s) for s in train_set]
y_train = [sent2labels(s) for s in train_set]

X_test = [sent2features(s) for s in test_set]
y_test = [sent2labels(s) for s in test_set]

print("Feature extraction COMPLETE!")

Extracting features...
Feature extraction COMPLETE!


In [10]:
# Train and Test
print("Training CRF model...")

# 1. Build the model
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)

# 2. Train it
crf.fit(X_train, y_train)
print("Training COMPLETE!")

# 3. Check Accuracy
y_pred = crf.predict(X_test)
accuracy = metrics.flat_accuracy_score(y_test, y_pred)
print(f"\nCRF Model Accuracy: {accuracy:.2%}")

# 4. RETEST: "Мама мыла раму"
# We have to process the sentence into features first
custom_sentence = "Мама мыла раму"
tokens = custom_sentence.split()
# Create dummy tags just to match the format [(word, tag), ...]
dummy_input = [(word, 'N/A') for word in tokens]
custom_features = [sent2features(dummy_input)] # Wrap in a list

prediction = crf.predict(custom_features)[0]

print("-" * 30)
print(f"Retesting: {custom_sentence}")
for word, tag in zip(tokens, prediction):
    print(f"{word} -> {tag}")

Training CRF model...
Training COMPLETE!

CRF Model Accuracy: 96.43%
------------------------------
Retesting: Мама мыла раму
Мама -> NOUN
мыла -> VERB
раму -> NOUN


In [12]:
from transformers import pipeline

print("Downloading Model...")

# Load pret-rained model
bert_tagger = pipeline(
    "token-classification",
    model="wietsedv/xlm-roberta-base-ft-udpos28-ru",
    aggregation_strategy="simple"
)

print("SUCCESS! Model loaded and ready for tagging.")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cpu


SUCCESS! Model loaded and ready for tagging.


In [13]:
# CELL: Calculate RoBERTa Accuracy (Scientific Method)
import torch
from tqdm import tqdm
from sklearn.metrics import accuracy_score

def evaluate_roberta_accuracy(test_data, pipeline):
    print(f"Evaluating RoBERTa on {len(test_data)} sentences...")
    print("This handles sub-word alignment automatically.")

    # Access the raw tools inside the pipeline
    tokenizer = pipeline.tokenizer
    model = pipeline.model
    device = model.device

    y_true = []
    y_pred = []

    # Map of RoBERTa ID numbers to Text Tags (e.g., 5 -> 'NOUN')
    id2label = model.config.id2label

    for sent in tqdm(test_data):
        # 1. Get the original words and tags
        # sent format: [[word, tag], [word, tag]...]
        words = [token[0] for token in sent]
        true_tags = [token[1] for token in sent]

        # 2. Tokenize with "is_split_into_words=True"
        # This keeps the mapping to our original list intact!
        inputs = tokenizer(words, is_split_into_words=True, return_tensors="pt").to(device)

        # 3. Get Predictions
        with torch.no_grad():
            logits = model(**inputs).logits

        # Get the most likely ID for each token
        predictions = torch.argmax(logits, dim=2)[0].cpu().numpy()

        # 4. Use word_ids to align
        # word_ids looks like: [None, 0, 0, 1, 2, 2, 2, None]
        # It tells us exactly which original word each token belongs to.
        word_ids = inputs.word_ids()

        aligned_preds = []
        previous_word_idx = None

        for i, word_idx in enumerate(word_ids):
            # Skip special tokens (None)
            if word_idx is None:
                continue

            # We only take the FIRST token of each word (Standard practice)
            if word_idx != previous_word_idx:
                tag_id = predictions[i]
                tag_name = id2label[tag_id]

                if "-" in tag_name:
                    tag_name = tag_name.split("-")[1]

                aligned_preds.append(tag_name)
                previous_word_idx = word_idx


        if len(true_tags) == len(aligned_preds):
            y_true.extend(true_tags)
            y_pred.extend(aligned_preds)

    # Statistics
    print("\n" + "="*40)
    print("FINAL ACCURACY")
    print("="*40)
    acc = accuracy_score(y_true, y_pred)
    print(f"RoBERTa Model Accuracy: {acc:.2%}")

    return acc

roberta_acc = evaluate_roberta_accuracy(test_set, bert_tagger)

Evaluating RoBERTa on 8800 sentences...
This handles sub-word alignment automatically.


100%|██████████| 8800/8800 [23:05<00:00,  6.35it/s]



FINAL ACCURACY
RoBERTa Model Accuracy: 96.42%


In [17]:
# The Final 4-Model POS Showdown with Smart Alignment
import pandas as pd
import torch

def get_roberta_pos_tags(original_tokens):
    # Access internal tools from the pipeline
    tokenizer = bert_tagger.tokenizer
    model = bert_tagger.model
    device = model.device
    id2label = model.config.id2label

    # 1. Tokenize the LIST of words (preserves alignment)
    inputs = tokenizer(original_tokens, is_split_into_words=True, return_tensors="pt").to(device)

    # 2. Predict
    with torch.no_grad():
        logits = model(**inputs).logits
    predictions = torch.argmax(logits, dim=2)[0].cpu().numpy()

    # 3. Align using word_ids (The magic fix)
    word_ids = inputs.word_ids()
    aligned_tags = []
    previous_word_idx = None

    for i, word_idx in enumerate(word_ids):
        if word_idx is None: continue # Skip [CLS]

        # Only take the tag for the FIRST piece of the word
        if word_idx != previous_word_idx:
            tag_id = predictions[i]
            label = id2label[tag_id]
            aligned_tags.append(label)
            previous_word_idx = word_idx

    return aligned_tags

def compare_four_models(sentences):
    # Header
    print(f"{'WORD':<15} {'UNI':<8} {'HMM':<8} {'CRF':<8} {'ROBERTA':<10}")
    print("=" * 65)

    for sent in sentences:
        print(f"\nInput: {sent}")
        print("-" * 65)
        tokens = sent.split()

        # 1. Unigram
        uni_tags = [tag if tag else 'None' for word, tag in unigram_tagger.tag(tokens)]

        # 2. HMM
        hmm_tags = [tag for word, tag in hmm_tagger.tag(tokens)]

        # 3. CRF
        dummy_input = [(word, 'N/A') for word in tokens]
        crf_features = [sent2features(dummy_input)]
        crf_tags = crf.predict(crf_features)[0]

        # 4. RoBERTa (Now using the smart function)
        roberta_tags = get_roberta_pos_tags(tokens)

        # Print Row
        for i, word in enumerate(tokens):
            u = uni_tags[i]
            h = hmm_tags[i]
            c = crf_tags[i]

            # Safety check for length (rare edge case)
            r = roberta_tags[i] if i < len(roberta_tags) else "???"

            print(f"{word:<15} {u:<8} {h:<8} {c:<8} {r:<10}")


final_sentences = [
    # 1. Slang/Morphology: "Googled" (zaguglil) -> VERB
    "Я загуглил ответ",

    # 2. Homonym: "Bake" (Verb) vs "Stove" (Noun)
    "Мама стала печь пироги , а в углу стояла печь",

    # 3. Proper Noun: "Yandex" (Capitalized) -> PROPN
    "Вчера Яндекс обновил поиск"
]

compare_four_models(final_sentences)

WORD            UNI      HMM      CRF      ROBERTA   

Input: Я загуглил ответ
-----------------------------------------------------------------
Я               PRON     PRON     PRON     PRON      
загуглил        None     VERB     VERB     VERB      
ответ           NOUN     NOUN     NOUN     NOUN      

Input: Мама стала печь пироги , а в углу стояла печь
-----------------------------------------------------------------
Мама            NOUN     NOUN     NOUN     NOUN      
стала           VERB     VERB     VERB     VERB      
печь            NOUN     NOUN     NOUN     VERB      
пироги          None     PROPN    NOUN     NOUN      
,               PUNCT    PUNCT    PUNCT    PUNCT     
а               CCONJ    CCONJ    CCONJ    CCONJ     
в               ADP      ADP      ADP      ADP       
углу            NOUN     NOUN     NOUN     NOUN      
стояла          VERB     VERB     VERB     VERB      
печь            NOUN     NOUN     VERB     NOUN      

Input: Вчера Яндекс обновил поис